# Phase 2 · Notebook 01 — HuggingFace Baseline (`dslim/bert-base-NER`)

In Phase 1 we used spaCy's strongest off-the-shelf English NER (`en_core_web_trf`) and got an overall partial-match F1 of ~0.57. The natural follow-up question is: **does swapping in a different general-purpose model help?**

[`dslim/bert-base-NER`](https://huggingface.co/dslim/bert-base-NER) is one of the most-downloaded NER models on the HuggingFace Hub. It's a `bert-base-cased` fine-tuned on CoNLL-2003 — so it predicts the four labels `PER`, `ORG`, `LOC`, `MISC`.

This notebook runs it through the same evaluation framework as Phase 1, against TAB's full test split. The expected finding: it'll close some of the spaCy gap on PERSON, but will be **even worse** on `DATETIME`, `QUANTITY`, `CODE`, and `DEM`, because those labels don't exist in CoNLL-2003. That's the empirical answer to *"can we just use a different general-purpose model?"*

---


In [ ]:
# ── Run me first if you're on Colab (skip locally — already in requirements.txt) ──
# !pip -q install transformers datasets evaluate seqeval accelerate \
#                 presidio-analyzer presidio-anonymizer scikit-learn spacy
# !python -m spacy download en_core_web_lg
# !python -m spacy download en_core_web_sm


## Setup


In [ ]:
import sys
sys.path.insert(0, "../src")

import time
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from anonymisation.data import load_tab
from anonymisation.evaluation import (
    evaluate_document, merge_results, results_to_dataframe,
)
from anonymisation.predictors import make_hf_predictor, DEFAULT_HF_LABEL_TO_TAB
from anonymisation.device import best_device, report_device

print(report_device())


## Configuration


In [ ]:
HF_MODEL = "dslim/bert-base-NER"
USE_FULL_TEST_SET = True
SAMPLE_SIZE = 100              # only used when USE_FULL_TEST_SET = False
MATCH_MODES = ["partial", "exact"]
RESULTS_PATH = "results/hf_results.csv"


## Load the model


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

device, _ = best_device()
print(f"Loading {HF_MODEL} on device={device} ...")
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL)
model = AutoModelForTokenClassification.from_pretrained(HF_MODEL)

# device for HF pipeline: -1 = CPU, 0 = first CUDA. MPS support in pipeline is
# patchy across transformers versions, so we fall back to CPU on MPS.
hf_device = 0 if device == "cuda" else -1
ner = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=hf_device,
)
predict = make_hf_predictor(ner)

# Smoke test
out = predict("Maria Petrova lives in Sofia.")
print("Sanity check:", out)


## Load TAB & run the evaluation


In [ ]:
dataset = load_tab()
test_docs = list(dataset["test"])
if not USE_FULL_TEST_SET:
    test_docs = test_docs[:SAMPLE_SIZE]
print(f"Evaluating on {len(test_docs)} TAB test documents.")


In [ ]:
all_merged = {}
for mode in MATCH_MODES:
    print(f"\n--- {mode} match ---")
    per_doc = []
    start = time.time()
    for i, doc in enumerate(test_docs):
        if (i + 1) % 50 == 0:
            elapsed = time.time() - start
            print(f"  {i + 1}/{len(test_docs)}   ({(i + 1)/elapsed:.1f} docs/s)")
        per_doc.append(evaluate_document(predict, doc, mode=mode))
    elapsed = time.time() - start
    print(f"  done in {elapsed:.1f}s")
    all_merged[mode] = merge_results(per_doc)

results_df = results_to_dataframe(all_merged)
results_df.insert(0, "model", "hf_bert_base_ner")
results_df.to_csv(RESULTS_PATH, index=False)
print(f"\nSaved → {RESULTS_PATH}")


## Results table


In [ ]:
from anonymisation.mapping import TAB_TO_SPACY

for mode in MATCH_MODES:
    merged = all_merged[mode]
    print(f"\n── {mode.upper()} MATCH ──")
    rows = []
    for et in list(TAB_TO_SPACY.keys()) + ["_ALL"]:
        r = merged[et]
        rows.append({
            "Entity": et if et != "_ALL" else "▶ OVERALL",
            "TP": r.tp, "FP": r.fp, "FN": r.fn,
            "Precision": f"{r.precision:.1%}",
            "Recall":    f"{r.recall:.1%}",
            "F1":        f"{r.f1:.1%}",
        })
    print(pd.DataFrame(rows).to_string(index=False))


## What to look for in the numbers

The CoNLL-2003 label set covers `PER / ORG / LOC / MISC`. Anything in TAB that doesn't map to one of these — `DATETIME`, `QUANTITY`, `CODE`, `DEM` — is **structurally invisible** to this model, so we expect 0% recall on those four entity types.

This is an instructive failure mode rather than a surprising one: it confirms that the gap we saw in Phase 1 is not a quirk of spaCy. Any general-purpose English NER will hit a wall on legal-domain categories. To close that gap you have to *train on the right labels* — which is what notebook 03 does.

The `MISC` category is also worth noting: HF's MISC will fire on a much wider population than TAB's MISC (works of art, events, products), which means we should expect high false-positive rates there.

We'll combine these results with Phase 1's spaCy numbers in `04_head_to_head.ipynb`.
